In [51]:
BASE_NAME = "liar"

In [61]:
# =========================
# 0) Auto-locate processed/ and create dataset folders
# =========================
from pathlib import Path
import os

def _find_processed_dir(start: Path, max_depth: int = 5) -> Path:
    # Common locations first
    common = [
        start / "processed",
        start / "data" / "processed",
    ]
    for p in common:
        if p.exists() and p.is_dir():
            return p.resolve()

    # Bounded BFS search (keeps it reasonably fast)
    start = start.resolve()
    q = [(start, 0)]
    seen = set()

    while q:
        cur, depth = q.pop(0)
        if cur in seen:
            continue
        seen.add(cur)

        cand = cur / "processed"
        if cand.exists() and cand.is_dir():
            return cand.resolve()

        if depth >= max_depth:
            continue

        try:
            for child in cur.iterdir():
                if child.is_dir() and not child.name.startswith("."):
                    # skip huge/common noise folders
                    if child.name.lower() in {"node_modules", ".git", "__pycache__", ".venv", "venv"}:
                        continue
                    q.append((child, depth + 1))
        except Exception:
            continue

    raise FileNotFoundError(
        f"Could not find a 'processed' folder starting from: {start} (max_depth={max_depth})."
    )

# Move 2 steps back from current working directory
cwd = Path.cwd().resolve()
repo_root_guess = cwd.parent.parent

PROCESSED_ROOT = _find_processed_dir(repo_root_guess)

# Create dataset folders if missing
DATASET_NAMES = ["_coaid_split_tmp", "_fakehealth_split_tmp", "_fakenewsnet_split_tmp", "_hover_split_tmp", "_liar_split_tmp"]
for name in DATASET_NAMES:
    (PROCESSED_ROOT / name).mkdir(parents=True, exist_ok=True)

print("cwd:", cwd)
print("repo_root_guess (2 steps back):", repo_root_guess)
print("PROCESSED_ROOT:", PROCESSED_ROOT)
print("Created/verified dataset folders:", ", ".join(DATASET_NAMES))


cwd: C:\Users\insoo\Documents\Western_WAI\explainable-misinfo-ai\data\unified_schema\After_datasets_ready
repo_root_guess (2 steps back): C:\Users\insoo\Documents\Western_WAI\explainable-misinfo-ai\data
PROCESSED_ROOT: C:\Users\insoo\Documents\Western_WAI\explainable-misinfo-ai\data\processed
Created/verified dataset folders: _coaid_split_tmp, _fakehealth_split_tmp, _fakenewsnet_split_tmp, _hover_split_tmp, _liar_split_tmp


In [53]:
PROCESSED_ROOT

WindowsPath('C:/Users/insoo/Documents/Western_WAI/explainable-misinfo-ai/data/processed')

In [54]:
# =========================
# 1) USER CONFIG
# =========================
from pathlib import Path

INPUT_PATH = Path(PROCESSED_ROOT / f"_{BASE_NAME}_tmp")

OUTPUT_DIR = Path(PROCESSED_ROOT / f"{BASE_NAME}")

USE_CONTENT_STATUS_HINTS = False


In [55]:
# =========================
# 2) Imports + constants
# =========================
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

EXPECTED_COLS = [
    'dataset',
    'id',
    'claim_text',
    'article_text',
    'content_status',
    'label_raw',
    'label',
    'label_confidence',
    'label_mode',
    'label_3way',
    'label_bin',
    'source_id',
    'claim_norm_hash',
    'lang',
    'content_char_len',
]

LABELS = ["false", "mixed", "true"]
VARIANTS = ["claimonly", "full"]

def _ensure_schema(df: pd.DataFrame) -> pd.DataFrame:
    # Add missing cols as NA
    for c in EXPECTED_COLS:
        if c not in df.columns:
            df[c] = pd.NA
    # Drop extra cols, enforce order
    return df[EXPECTED_COLS].copy()

def _load_any_parquet(path: Path) -> pd.DataFrame:
    if path.is_dir():
        files = sorted(path.glob("*.parquet"))
        if not files:
            raise FileNotFoundError(f"No parquet files found in directory: {path}")
        dfs = []
        for f in tqdm(files, desc="Reading parquets"):
            dfs.append(pd.read_parquet(f))
        return pd.concat(dfs, ignore_index=True)
    else:
        return pd.read_parquet(path)

def _normalize_label_value(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return None
    # numeric codes
    if isinstance(v, (int, np.integer)):
        if v == 0: return "false"
        if v == 1: return "mixed"
        if v == 2: return "true"
        return None
    # numeric in string
    if isinstance(v, str):
        s = v.strip().lower()
        if s in {"0", "false"}: return "false"
        if s in {"1", "mixed"}: return "mixed"
        if s in {"2", "true"}:  return "true"
        # common alternates
        if s in {"supported"}: return "true"
        if s in {"not_supported", "not supported"}: return "false"
        if s in {"partially_supported", "partially supported"}: return "mixed"
        return None
    # other numeric types
    try:
        iv = int(v)
        return _normalize_label_value(iv)
    except Exception:
        return None

def _get_label_group(row) -> str | None:
    # Prefer label_3way, fallback to label
    v = row.get("label_3way", pd.NA)
    out = _normalize_label_value(v)
    if out is not None:
        return out
    v2 = row.get("label", pd.NA)
    return _normalize_label_value(v2)

def _has_article_text(row) -> bool:
    txt = row.get("article_text", None)
    if isinstance(txt, str) and txt.strip():
        return True
    if USE_CONTENT_STATUS_HINTS:
        cs = row.get("content_status", None)
        if isinstance(cs, str) and cs.strip().lower() in {"success", "archived_success"}:
            # If content_status says success but article_text is empty, still treat as claimonly
            return False
    return False


In [56]:
# =========================
# 3) Load data
# =========================
df_raw = _load_any_parquet(INPUT_PATH)

print("Loaded rows:", len(df_raw))
print("Loaded cols:", len(df_raw.columns))

df = _ensure_schema(df_raw)
print("Schema enforced. Cols now:", list(df.columns))


Reading parquets: 100%|██████████| 3/3 [00:00<00:00, 35.67it/s]

Loaded rows: 12791
Loaded cols: 15
Schema enforced. Cols now: ['dataset', 'id', 'claim_text', 'article_text', 'content_status', 'label_raw', 'label', 'label_confidence', 'label_mode', 'label_3way', 'label_bin', 'source_id', 'claim_norm_hash', 'lang', 'content_char_len']


In [57]:
# =========================
# 4) Compute label_group + variant
# =========================
# Vectorized-ish approach: apply row-wise for robustness (schema already enforced)
label_group = []
variant = []

for r in tqdm(df.to_dict("records"), desc="Label/variant"):
    lg = _get_label_group(r)
    label_group.append(lg)
    has_art = _has_article_text(r)
    variant.append("full" if has_art else "claimonly")

df["label_group"] = label_group
df["variant"] = variant

print(df["label_group"].value_counts(dropna=False))
print(df["variant"].value_counts(dropna=False))


Label/variant: 100%|██████████| 12791/12791 [00:00<00:00, 1029243.98it/s]

label_group
mixed    7184
false    3554
true     2053
Name: count, dtype: int64
variant
claimonly    12791
Name: count, dtype: int64


In [58]:
# =========================
# 5) Write outputs (6 parquet files)
# =========================
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

written = []
summary_rows = []

for lg in LABELS:
    for v in VARIANTS:
        out_path = OUTPUT_DIR / f"{BASE_NAME}_{lg}_{v}.parquet"
        sub = df[(df["label_group"] == lg) & (df["variant"] == v)].copy()

        # Enforce the strict schema in outputs (and only those fields)
        sub = _ensure_schema(sub)

        sub.to_parquet(out_path, index=False)
        written.append(out_path)
        summary_rows.append({"label": lg, "variant": v, "rows": len(sub), "path": str(out_path)})

pd.DataFrame(summary_rows)


,label,variant,rows,path
0,false,claimonly,3554,C:\Users\insoo\Documents\Western_WAI\explainab...
1,false,full,0,C:\Users\insoo\Documents\Western_WAI\explainab...
2,mixed,claimonly,7184,C:\Users\insoo\Documents\Western_WAI\explainab...
3,mixed,full,0,C:\Users\insoo\Documents\Western_WAI\explainab...
4,true,claimonly,2053,C:\Users\insoo\Documents\Western_WAI\explainab...
5,true,full,0,C:\Users\insoo\Documents\Western_WAI\explainab...


In [59]:
# =========================
# 6) Quick sanity checks
# =========================
# (a) All output files exist?
missing = [p for p in written if not p.exists()]
print("Missing outputs:", missing)

# (b) Read one file and confirm schema/order
sample_path = written[0]
sample = pd.read_parquet(sample_path)
print("Sample:", sample_path.name)
print("Cols match expected:", list(sample.columns) == EXPECTED_COLS)
print("Rows:", len(sample))


Missing outputs: []
Sample: liar_false_claimonly.parquet
Cols match expected: True
Rows: 3554
